# Project 2 — Notebook 01: Data Cleaning & EDA

**Infotact Data Analytics Internship — Cohort Retention & CLTV Analysis**

**Week 1 goal:** turn the raw Online Retail transaction data into one clean table we can trust.

In this notebook I:
1. load the local dataset,
2. standardize the column names,
3. check the main data quality problems,
4. remove cancelled/refunded/invalid transactions,
5. remove rows without customer IDs for cohort work,
6. save a short cleaning summary.

The raw data stays inside `data/`, which is ignored by git.

## Step 1 — Import the tools

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)

import pandas as pd
import numpy as np

from src.data_prep import load_combined_dataset, clean_dataset

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)
print('pandas version:', pd.__version__)

## Step 2 — Load the raw data

The helper function accepts either `data/Online_Retail.xlsx`, `data/Online_Retail.csv`, or the older three split files. It returns one standard table with simple lower-case column names.

In [ ]:
combined = load_combined_dataset(DATA_DIR)

print('combined rows :', f'{len(combined):,}')
print('date range    :', combined['invoice_datetime'].min(), '->', combined['invoice_datetime'].max())
print('columns       :', list(combined.columns))
combined.head(3)

## Step 3 — Check data quality before cleaning

I want to measure the problems before dropping rows. This makes the cleaning decision easier to explain later.

In [ ]:
quality_checks = {
    'total_rows': len(combined),
    'missing_customer_id': combined['customer_id'].isna().sum(),
    'missing_description': combined['description'].isna().sum(),
    'cancelled_invoices': combined['invoice_no'].str.startswith('C', na=False).sum(),
    'negative_quantity_rows': (combined['quantity'] < 0).sum(),
    'zero_or_negative_price_rows': (combined['unit_price'] <= 0).sum(),
    'duplicate_rows': combined.duplicated().sum(),
}

for name, value in quality_checks.items():
    print(name, ':', f'{value:,}')

## Step 4 — Clean the data with an audit trail

Cleaning rules:
- remove exact duplicate rows,
- remove cancelled invoices (`InvoiceNo` starts with `C`),
- keep only positive quantity,
- keep only positive unit price,
- require `customer_id`, because retention needs a known customer.

I do it step-by-step so I can see how many rows each rule removed.

In [ ]:
audit = []

def log_step(step_name, df):
    audit.append({'step': step_name, 'rows_remaining': len(df)})

work = combined.copy()
log_step('0. combined start', work)

work = work.drop_duplicates()
log_step('1. drop duplicate rows', work)

work = work[~work['invoice_no'].str.startswith('C', na=False)]
log_step('2. remove cancelled invoices', work)

work = work[work['quantity'] > 0]
log_step('3. keep quantity > 0', work)

work = work[work['unit_price'] > 0]
log_step('4. keep unit_price > 0', work)

work = work[work['customer_id'].notna()]
log_step('5. require customer_id present', work)

work = work.copy()
work['line_total'] = (work['quantity'] * work['unit_price']).round(2)

audit_df = pd.DataFrame(audit)
audit_df['rows_removed'] = audit_df['rows_remaining'].shift(1) - audit_df['rows_remaining']
audit_df

## Step 5 — Final cleaned dataset checks

In [ ]:
clean = clean_dataset(combined)

print('FINAL CLEANED DATASET')
print('rows            :', f'{len(clean):,}')
print('unique customers:', f'{clean["customer_id"].nunique():,}')
print('unique invoices :', f'{clean["invoice_no"].nunique():,}')
print('date range      :', clean['invoice_datetime'].min(), '->', clean['invoice_datetime'].max())
print('countries       :', clean['country'].nunique())
print('total revenue   : £', f'{clean["line_total"].sum():,.2f}')
clean.head(3)

## Step 6 — Save a local cleaned file and a small summary

In [ ]:
# This CSV is local only because data/ is git-ignored.
clean.to_csv(DATA_DIR / 'cleaned_transactions.csv', index=False)

audit_df.to_csv(OUTPUT_DIR / 'cleaning_audit.csv', index=False)

print('Saved local clean data to:', DATA_DIR / 'cleaned_transactions.csv')
print('Saved audit table to     :', OUTPUT_DIR / 'cleaning_audit.csv')

## Week 1 notes

The data is now clean enough for cohort and CLTV work. Rows without `customer_id` were removed only because this project tracks customer behavior over time. If the business wanted total sales reporting, those anonymous rows could be analyzed separately.